# Sultai mana-base explorer

Run all cells once. Then edit **Shock penalty** and **Town penalty** in the parameter cell and rerun the analysis cell below it. The expensive draw calculations stay cached in the tables.

Each penalty is charged per matching land in the base; Cavern has no composition penalty. The starting values are illustrative, not calibrated life costs. Modes have uniform weights within each physical spell profile.

Basic and nonbasic bounds are editable below. Exhaustive builds use the batch backend without grid-size cutoffs. The candidate count and capacity-grid size are printed before building. Town is flexible; four Caverns and one Swamp are fixed within the 26-land total. Cavern has no penalty.

The report scores the full pool once, selects only the top ten rows, and reuses the score snapshot for casting-cost and penalty crossings.

V1 checks colors among all lands seen, multiplied by a separate independent availability factor: Town is fully available through T3, then has per-land rate (T-1)/T. Optional families `fast`, `slow`, and `tapped` can be enabled explicitly in `LAND_FAMILIES`; they have no basic land types. Rebuild the table for these model changes. The weighted batch grid uses additional memory; inspect pool sizes before building.

The search uses the Standard legality snapshot dated 2026-11-03 in `mana_lands`: Sultai fast lands include Blooming Marsh and Botanical Sanctum, excluding Darkslick Shores. This is an explicit catalog snapshot; update it when legality changes.


For the supplied deck with **0?2 Opulent Palaces**, use `sultai_local_search.ipynb` or `python -m examples.sultai_optimize`. This exhaustive explorer keeps its separate flexible pool without tri-lands; enabling them increases grid memory substantially.

In [ ]:
from pathlib import Path
import sys
from time import perf_counter
from math import prod
from html import escape

# Works when launched from the repository or one of its subdirectories.
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'mana_opportunities.py').exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from IPython.display import HTML, display
from tqdm.auto import tqdm
from mana_lands import ManaSource
from mana_opportunities import CastingMode, Spell, SpellCost, spell_components
from mana_search import ManaSpace, build_component_table


def show_table(headers, rows):
    header = ''.join(f'<th>{escape(str(h))}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(v))}</td>' for v in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>'))


def source_name(source):
    return f'{source.kind}:{source.colors}'


def source_list(entries):
    return ', '.join(f'{n} {source_name(source)}' for source, n in entries) or 'none'


def number(value):
    return 'unbounded' if value is None else f'{float(value):.8g}'


def sensitivity_row(result):
    interval = result.optimal_interval
    below, above = result.lower, result.upper
    return (
        result.parameter, number(result.current_value), number(interval.lower),
        source_list(below.swap.incoming) if below else 'none',
        source_list(below.swap.out) if below else 'none',
        number(interval.upper),
        source_list(above.swap.incoming) if above else 'none',
        source_list(above.swap.out) if above else 'none',
    )


## Spell profiles and search space

Edit these definitions here, then rerun this cell, build the search tables, and run the analysis. `LAND_COUNT`, `COLORS`, and `LAND_FAMILIES` independently define the search pool. Each enabled family receives per-source bounds; only on-color basics and duals are selected, while rainbow sources retain all colors. Mode weights default to uniform within each spell profile.

The shocks explicitly carry both basic land types. Verges, Town, and Cavern carry none. Verge secondary colors unlock when an enabling land type appears anywhere among lands seen, even if that land would not fit within the turn's land drops. This deliberately optimistic approximation avoids sequencing and battlefield search. Verges have no additional composition penalty.

`FIXED_LANDS` gives exact deck counts and can introduce sources whose families are omitted from `LAND_FAMILIES`. These cards occupy five of the 26 land slots. They are still drawn probabilistically; fixing a deck count does not put the land in every hand.


In [ ]:
from examples.sultai import SPELLS

# Complete bounded search: explicit land count, colors, and per-source family bounds.
LAND_COUNT = 26
COLORS = "UBG"
FORMAT = "standard"
LAND_FAMILIES = {
    "basic": (0, 1),
    "shock": (0, 4),
    "fast": (0, 4),
    "verge": (0, 4),
    "town": (0, 4),
}

# Included in LAND_COUNT: optimize the remaining 21 lands.
FIXED_LANDS = {
    ManaSource("WUBRG", "cavern"): 4,
    ManaSource("B", "basic"): 1,
}


## Build search tables

Set `TURNS` below to a positive integer (default 7). Opportunities are averaged from turn 1 through `TURNS`, over both play and draw positions. Turns are averaged during construction, so changing this setting requires rebuilding the table and rerunning the analysis.

Run this cell again only if you change the spell profile, bounds, or evaluation settings. The explicit search pool includes basics, typed shocks, Town/Cavern rainbow sources, and conditional Verges. Cavern restrictions are excluded.

Batch creation shares probability calculations across every base. Component bars report the expensive transforms; base bars report row assembly. The scalar backend remains available with `backend="scalar"` for exhaustive reference checks.


In [ ]:

TURNS = 7  # Evaluate turns 1 through TURNS; changing this requires rebuilding.

space = ManaSpace.from_colors(
    LAND_COUNT, COLORS, families=LAND_FAMILIES, format=FORMAT, fixed=FIXED_LANDS,
)
show_table(["Fixed source", "Count"], [(source_name(s), n) for s, n in FIXED_LANDS.items()])
print(f"Total lands: {LAND_COUNT}; fixed: {sum(FIXED_LANDS.values())}; flexible: {LAND_COUNT - sum(FIXED_LANDS.values())}")
print(f'Exhaustive candidates: {space.candidate_count()}')
grid_cells = prod(hi + 1 for _, hi in space.bounds.values())
print(f'Capacity grid: {grid_cells:,} cells')
started = perf_counter()
table = build_component_table(
    space, SPELLS, turns=TURNS,
    progress=lambda candidates: tqdm(candidates, total=space.candidate_count(), desc='Base rows', unit='base'),
    component_progress=lambda costs: tqdm(costs, total=len(spell_components(SPELLS)[1]),
                                          desc='Components', unit='cost'),
)
print(f'{len(table.rows)} bases built for turns 1–{TURNS} (play/draw) in {perf_counter() - started:.2f}s')


## Parameters

Edit these two values, run this cell, then run the analysis cell. Zero disables that penalty. Changing these values does not require rebuilding the tables.


In [ ]:
shock_penalty = 0.012
town_penalty = 0.015  # Independent control; initially equal to shock_penalty.


## Analysis

The optimum appears first, followed by the ten highest-scoring bases. Cost-weight sensitivities follow, then penalty sensitivities.

Each sensitivity varies one absolute nonnegative parameter with all others fixed. The optimum stays optimal between the lower and upper boundaries (including ties). IN/OUT below or above a boundary describes the immediate successor relative to this optimum. A zero lower boundary has no transition into negative values. Exact fractions remain available in `analysis`.


In [ ]:
penalties = dict(shock_penalty=shock_penalty, town_penalty=town_penalty)
started = perf_counter()
scored = table.score(**penalties)
ranked = scored.rank(limit=10)
optimum = ranked[0]
winner = dict(zip(space.sources, optimum.counts))

display(HTML('<h3>Optimum solution</h3>'))
print(f'{len(table.rows):,} candidates; shock penalty {shock_penalty}; town penalty {town_penalty}')
show_table(
    ['Raw opportunities', 'Penalty', 'Score'],
    [(f'{float(optimum.raw_score):.6f}', f'{float(optimum.penalty):.6f}',
      f'{float(optimum.score):.6f}')],
)
show_table(['Source', 'Count'],
           [(source_name(s), count) for s, count in winner.items() if count] + [('Total', sum(winner.values()))])

display(HTML('<h3>Top 10 candidates</h3>'))
show_table(
    ['Rank', 'Raw opportunities', 'Penalty', 'Score'],
    [(i, f'{float(r.raw_score):.6f}', f'{float(r.penalty):.6f}', f'{float(r.score):.6f}')
     for i, r in enumerate(ranked, 1)],
)

display(HTML('<h3>Casting-cost weight sensitivity</h3>'))
cost_sensitivities = {cost: scored.crossings(cost) for cost in table.costs}
sensitivity_headers = ['Parameter', 'Current', 'Lower boundary', 'IN below', 'OUT below',
                       'Upper boundary', 'IN above', 'OUT above']
show_table(sensitivity_headers, [sensitivity_row(result) for result in cost_sensitivities.values()])

display(HTML('<h3>Penalty sensitivity</h3>'))
penalty_sensitivities = {parameter: scored.crossings(parameter) for parameter in penalties}
show_table(sensitivity_headers, [sensitivity_row(result) for result in penalty_sensitivities.values()])
analysis = dict(scored=scored, optimum=optimum, ranked=ranked,
                cost_sensitivities=cost_sensitivities, penalty_sensitivities=penalty_sensitivities)
print(f'Report computed in {perf_counter() - started:.2f}s')
